In [ ]:
import glob
import time
import pandas as pd
from imblearn.over_sampling import RandomOverSampler, SMOTE
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
file_path = glob.glob('**/ml_features_and_labels.csv', recursive=True)[0]
df = pd.read_csv(file_path)
df.head()

In [ ]:
feature_columns = [col for col in df.columns if col not in ['label', 'split', 'taxonomy', 'ID']]
X = df[feature_columns].replace({True: 1, False: 0, 'TRUE': 1, 'FALSE': 0})
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
y = df['label']

X_train = X[df['split'] == 'train']
y_train = y[df['split'] == 'train']
X_val = X[df['split'] == 'val']
y_val = y[df['split'] == 'val']
X_test = X[df['split'] == 'test']
y_test = y[df['split'] == 'test']

In [ ]:
model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss')

start = time.perf_counter()

# Training
model.fit(X_train, y_train)

end = time.perf_counter()
training_time = end - start
print(f'Training time: {training_time:.2f} seconds')

In [ ]:
start = time.perf_counter()
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)
end = time.perf_counter()
testing_time = end - start

results = pd.DataFrame({
    'Split': ['Validation', 'Test'],
    'Accuracy': [accuracy_score(y_val, y_val_pred), accuracy_score(y_test, y_test_pred)],
    'Precision': [precision_score(y_val, y_val_pred, zero_division=0), precision_score(y_test, y_test_pred, zero_division=0)],
    'Recall': [recall_score(y_val, y_val_pred, zero_division=0), recall_score(y_test, y_test_pred, zero_division=0)],
    'F1 Score': [f1_score(y_val, y_val_pred, zero_division=0), f1_score(y_test, y_test_pred, zero_division=0)]
})

print(f'Testing time: {testing_time:.2f} seconds')
print('\nTest classification report:')
print(classification_report(y_test, y_test_pred, zero_division=0))
results